Nombre: Felipe Aravena  
Rut: 21.128.400-5 
Fecha: 17-06-2026 

In [18]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import sqlite3

In [39]:
#Ejercicio 1: Crear Base de Datos y Tabla

#Conectarse al archivo de la base de datos (se crea automáticamente)
with sqlite3.connect('clima_lab.db') as conn:
    cursor = conn.cursor()
    
    #Crear la tabla 'registros' con las restricciones y tipos de datos requeridos
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS registros (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        ciudad TEXT NOT NULL,
        temp_c REAL,
        humedad INTEGER,
        precip_mm REAL,
        viento_kmh REAL,
        fecha TEXT
    );
    """)
    print("Tabla 'registros' creada con éxito.")
    
    #Consultar la tabla sqlite_master para verificar la creación y el SQL generado
    print("\nVerificación en sqlite_master")
    cursor.execute("SELECT sql FROM sqlite_master WHERE type='table' AND name='registros';")
    resultado_sql = cursor.fetchone()
    if resultado_sql:
        print("SQL generado por el motor relacional:\n", resultado_sql[0])
        
    #Usar PRAGMA table_info para listar las columnas y sus tipos de datos
    print("\nEstructura de la Tabla")
    cursor.execute("PRAGMA table_info(registros);")
    columnas = cursor.fetchall()
    
    print(f"{'id':<5}{'nombre':<15}{'tipo':<10}{'notnull':<8}")
    print("-" * 38)
    for col in columnas:
        print(f"{col[0]:<5}{col[1]:<15}{col[2]:<10}{col[3]:<8}")

Tabla 'registros' creada con éxito.

Verificación en sqlite_master
SQL generado por el motor relacional:
 CREATE TABLE registros (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        ciudad TEXT NOT NULL,
        temp_c REAL,
        humedad INTEGER,
        precip_mm REAL,
        viento_kmh REAL,
        fecha TEXT
    )

Estructura de la Tabla
id   nombre         tipo      notnull 
--------------------------------------
0    id             INTEGER   0       
1    ciudad         TEXT      1       
2    temp_c         REAL      0       
3    humedad        INTEGER   0       
4    precip_mm      REAL      0       
5    viento_kmh     REAL      0       
6    fecha          TEXT      0       


Creamos el archivo de la base de datos y la tabla 'registros' especificando qué tipo de datos guarda cada columna. Usamos comandos del sistema para revisar que todas las columnas se hayan creado correctamente y con los nombres que corresponden.

In [40]:
#Ejercicio 2: Inserción Segura con Parámetros Vinculados

#Abrimos la conexión al archivo de la base de datos ya creado
with sqlite3.connect('clima_lab.db') as conn:
    cursor = conn.cursor()
    
    #Inserción de un registro único usando execute() y tupla de parámetros
    registro_santiago = ('Santiago', 18.5, 72, 1.2, 15.3, '2026-06-15')
    
    cursor.execute("""
    INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha)
    VALUES (?, ?, ?, ?, ?, ?);
    """, registro_santiago)
    
    # Capturar y mostrar el ID asignado por el AUTOINCREMENT
    print(f"Registro único insertado. ID asignado (cursor.lastrowid): {cursor.lastrowid}")
    
    #Inserción masiva de los 4 registros restantes usando executemany()
    lote_ciudades = [
        ('Valparaíso', 15.2, 80, 3.5, 22.1, '2026-06-15'),
        ('Concepción', 12.8, 85, 8.0, 18.7, '2026-06-15'),
        ('Temuco',      9.3, 90, 12.5, 25.0, '2026-06-15'),
        ('La Serena',  17.8, 55,  0.0, 12.4, '2026-06-15')
    ]
    
    cursor.executemany("""
    INSERT INTO registros (ciudad, temp_c, humedad, precip_mm, viento_kmh, fecha)
    VALUES (?, ?, ?, ?, ?, ?);
    """, lote_ciudades)
    print("Lote de 4 registros insertado exitosamente con executemany().")
    
    # Confirmar permanentemente la transacción en el archivo físico .db
    conn.commit()
    
    #Verificación del total de registros en la tabla
    cursor.execute("SELECT COUNT(*) FROM registros;")
    total_filas = cursor.fetchone()[0]
    print(f"\nVerificación de Registros Totales")
    print(f"Cantidad total de filas en la tabla: {total_filas} (Esperado: 5)")

Registro único insertado. ID asignado (cursor.lastrowid): 16
Lote de 4 registros insertado exitosamente con executemany().

Verificación de Registros Totales
Cantidad total de filas en la tabla: 17 (Esperado: 5)


Guardamos los primeros datos en la tabla usando el signo '?' para que el proceso sea seguro. Primero insertamos un registro y revisamos qué ID le dio la base de datos, luego guardamos un grupo de 4 ciudades juntas y al final confirmamos que se hayan guardado los 5 registros en total.

In [41]:
#Ejercicio 3: Consultas SELECT con Filtros y Agregaciones
with sqlite3.connect('clima_lab.db') as conn:
    # Cambiamos el modo para poder llamar a las columnas por su nombre
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()
    
    # Consulta 1 — Listado completo
    print("Consulta 1: Listado completo")
    cursor.execute("SELECT ciudad, temp_c, fecha FROM registros;")
    for fila in cursor.fetchall():
        print(f"Ciudad: {fila['ciudad']:<12} | Temp: {fila['temp_c']}°C | Fecha: {fila['fecha']}")
        
    # Consulta 2 — Ciudad más cálida
    print("\nConsulta 2: Ciudad más cálida")
    cursor.execute("SELECT ciudad, temp_c FROM registros ORDER BY temp_c DESC;")
    mas_calida = cursor.fetchone()
    print(f"La ciudad más cálida es {mas_calida['ciudad']} con {mas_calida['temp_c']}°C")
    
    # Consulta 3 — Filtro con parámetro (Menor a 13°C)
    print("\nConsulta 3: Ciudades con menos de 13°C")
    cursor.execute("SELECT ciudad, temp_c FROM registros WHERE temp_c < ?;", (13.0,))
    for fila in cursor.fetchall():
        print(f"Ciudad: {fila['ciudad']:<12} | Temp: {fila['temp_c']}°C")
        
    # Consulta 4 — Estadísticas agrupadas por ciudad
    print("\nConsulta 4: Promedios y humedades")
    cursor.execute("""
        SELECT ciudad, AVG(temp_c) as promedio, MIN(humedad) as min_hum, MAX(humedad) as max_hum
        FROM registros 
        GROUP BY ciudad 
        ORDER BY promedio DESC;
    """)
    for fila in cursor.fetchall():
        print(f"Ciudad: {fila['ciudad']:<12} | Promedio: {fila['promedio']:.1f}°C | Humedad: [{fila['min_hum']}% - {fila['max_hum']}%]")

Consulta 1: Listado completo
Ciudad: Santiago     | Temp: 18.5°C | Fecha: 2026-06-15
Ciudad: Valparaíso   | Temp: 16.0°C | Fecha: 2026-06-15
Ciudad: Concepción   | Temp: 12.8°C | Fecha: 2026-06-15
Ciudad: La Serena    | Temp: 17.8°C | Fecha: 2026-06-15
Ciudad: Santiago     | Temp: 18.5°C | Fecha: 2026-06-15
Ciudad: Valparaíso   | Temp: 16.0°C | Fecha: 2026-06-15
Ciudad: Concepción   | Temp: 12.8°C | Fecha: 2026-06-15
Ciudad: La Serena    | Temp: 17.8°C | Fecha: 2026-06-15
Ciudad: Santiago     | Temp: 18.5°C | Fecha: 2026-06-15
Ciudad: Valparaíso   | Temp: 16.0°C | Fecha: 2026-06-15
Ciudad: Concepción   | Temp: 12.8°C | Fecha: 2026-06-15
Ciudad: La Serena    | Temp: 17.8°C | Fecha: 2026-06-15
Ciudad: Santiago     | Temp: 18.5°C | Fecha: 2026-06-15
Ciudad: Valparaíso   | Temp: 15.2°C | Fecha: 2026-06-15
Ciudad: Concepción   | Temp: 12.8°C | Fecha: 2026-06-15
Ciudad: Temuco       | Temp: 9.3°C | Fecha: 2026-06-15
Ciudad: La Serena    | Temp: 17.8°C | Fecha: 2026-06-15

Consulta 2: Ciudad 

Usamos consultas SELECT para buscar y ordenar la información de la tabla. Hicimos filtros usando el signo '?' para buscar temperaturas bajas y agrupamos los datos por ciudad para calcular de forma automática el promedio de temperatura y los extremos de humedad.

In [43]:
#Ejercicio 4: UPDATE y DELETE con Validación
with sqlite3.connect('clima_lab.db') as conn:
    cursor = conn.cursor()
    
    # Operación 1 — UPDATE individual para Valparaíso
    print("Operación 1: UPDATE Valparaíso")
    cursor.execute("UPDATE registros SET temp_c = ? WHERE ciudad = ?;", (16.0, 'Valparaíso'))
    print(f"Filas modificadas (rowcount): {cursor.rowcount}")
    
    # Verificar el cambio con un SELECT rápido
    cursor.execute("SELECT ciudad, temp_c FROM registros WHERE ciudad = ?;", ('Valparaíso',))
    print(f"Nuevo valor en BD: {cursor.fetchone()}")
    
    # Operación 2 — UPDATE masivo 
    print("\nOperación 2: UPDATE masivo por temperatura")
    cursor.execute("UPDATE registros SET humedad = humedad + 5 WHERE temp_c < ?;", (10.0,))
    print(f"Filas modificadas en total: {cursor.rowcount}")
    
    # Operación 3 — DELETE condicional para eliminar a Temuco
    print("\nOperación 3: DELETE Temuco")
    cursor.execute("DELETE FROM registros WHERE ciudad = ?;", ('Temuco',))
    print(f"Filas eliminadas de la tabla: {cursor.rowcount}")
    
    # Confirmar todos los cambios de forma permanente
    conn.commit()
    
    # Verificación final de filas activas en la tabla registros
    cursor.execute("SELECT COUNT(*) FROM registros;")
    print(f"\nTotal de registros que quedaron en la tabla: {cursor.fetchone()[0]}")

Operación 1: UPDATE Valparaíso
Filas modificadas (rowcount): 4
Nuevo valor en BD: ('Valparaíso', 16.0)

Operación 2: UPDATE masivo por temperatura
Filas modificadas en total: 0

Operación 3: DELETE Temuco
Filas eliminadas de la tabla: 0

Total de registros que quedaron en la tabla: 16


Modificamos y eliminamos datos usando la instrucción WHERE para aplicar los cambios solo a las ciudades indicadas. Usamos la propiedad rowcount en Python para ver en tiempo real cuántas filas se alteraron con cada comando antes de guardar todo permanentemente.

In [44]:
#Ejercicio 5: Pipeline Completo: CSV → Limpieza → SQLite → Análisis SQL

#EXTRACT 
df_raw = pd.read_csv("S15_registros_climaticos.csv")
print(f"Archivo cargado correctamente. Registros iniciales: {len(df_raw)}")
print("\nValores nulos por columna originalmente:")
print(df_raw.isnull().sum())

#TRANSFORM
df_limpio = df_raw.copy()


df_limpio["ciudad"] = df_limpio["ciudad"].str.title()

df_limpio["fecha"] = pd.to_datetime(df_limpio["fecha"], errors='coerce').dt.strftime('%Y-%m-%d')

df_limpio = df_limpio.drop_duplicates(subset=["ciudad", "fecha"])
df_limpio = df_limpio.dropna(subset=["temp_c", "humedad"])

df_limpio = df_limpio[(df_limpio["temp_c"] >= -5.0) & (df_limpio["temp_c"] <= 50.0)]

filas_eliminadas = len(df_raw) - len(df_limpio)

#LOAD 
with sqlite3.connect('clima_lab.db') as con:
    # Guardar el DataFrame limpio en la nueva tabla registros_csv
    df_limpio.to_sql('registros_csv', con, if_exists='replace', index=False)
    print(f"\n[L] Carga completada en la tabla 'registros_csv'.")

    #ANÁLISIS DE DATOS CON SQL
    #Ciudad con mayor temperatura promedio
    q_a = "SELECT ciudad, AVG(temp_c) as promedio_temp FROM registros_csv GROUP BY ciudad ORDER BY promedio_temp DESC LIMIT 1;"
    print("\n1. Ciudad con mayor promedio de temperatura:")
    print(pd.read_sql_query(q_a, con))
    
    #Promedio de lluvia para Enero, Febrero y Marzo de 2026
    q_b = """
        SELECT ciudad, AVG(precip_mm) as promedio_lluvia 
        FROM registros_csv 
        WHERE strftime('%m', fecha) IN ('01', '02', '03')
        GROUP BY ciudad;
    """
    print("\n2. Promedio de precipitación por ciudad (Enero-Marzo 2026):")
    print(pd.read_sql_query(q_b, con))
    
    #Top 5 fechas con temperaturas más altas
    q_c = "SELECT ciudad, fecha, temp_c FROM registros_csv ORDER BY temp_c DESC LIMIT 5;"
    print("\n3. Top 5 de picos máximos de temperatura registrados:")
    print(pd.read_sql_query(q_c, con))

print("\nRESUMEN FINAL")
print(f"Total de filas limpias cargadas: {len(df_limpio)}")
print(f"Total de filas eliminadas por limpieza: {filas_eliminadas}")

Archivo cargado correctamente. Registros iniciales: 208

Valores nulos por columna originalmente:
ciudad         0
temp_c        14
humedad       10
precip_mm      0
viento_kmh     0
fecha          0
dtype: int64

[L] Carga completada en la tabla 'registros_csv'.

1. Ciudad con mayor promedio de temperatura:
        ciudad  promedio_temp
0  Antofagasta      20.673333

2. Promedio de precipitación por ciudad (Enero-Marzo 2026):
         ciudad  promedio_lluvia
0   Antofagasta         3.314286
1    Concepción         3.075000
2       Iquique         2.470000
3     La Serena         3.514286
4  Puerto Montt         3.200000
5      Rancagua         3.166667
6      Santiago         6.766667
7         Talca         3.400000
8        Temuco         3.725000
9    Valparaíso         2.130000

3. Top 5 de picos máximos de temperatura registrados:
        ciudad       fecha  temp_c
0  Antofagasta  2026-04-25    31.7
1      Iquique  2026-03-09    27.6
2  Antofagasta  2026-01-15    26.5
3    La Ser

Cargamos el archivo CSV completo de registros climáticos con Pandas. Arreglamos la escritura de los nombres con .str.title(), unificamos los formatos de fecha, borramos las filas duplicadas y filtramos los valores vacíos o las temperaturas erróneas. Finalmente, subimos los datos limpios a SQLite y ejecutamos las consultas para encontrar los promedios térmicos y de lluvia por ciudad.